# PATSTAT Sleeping Beauty (Ke et al. 2015) — beauty coefficient B + awakening time T

The twin of `PatentView/notebook/patent_sb.ipynb`, same kernel verbatim. For every cited application the yearly
citation histogram `C[age]` (`age = citing filing year - cited filing year`, `>= 0`, `ps.EDGE_WHERE`) gives
`SB_B` and `SB_T`. Citation rows are counted (as in PatentView), not distinct citing applications.

## Output
`PATSTAT/output/patstat_sb.parquet` — `appln_id, SB_B, SB_T, n_cite`.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_sb.parquet')
ps.preflight('patstat_sb')

REF = ps.out('patstat_reference.parquet')

In [ ]:
from numba import njit, prange

@njit
def _sb_one(C):
    """C[age] = #citations received at that age (0..len-1). Returns (B, T). Mirrors MAG-SB.ipynb."""
    m = len(C)
    t_m = 0; cmax = C[0]
    for t in range(m):
        if C[t] > cmax:
            cmax = C[t]; t_m = t
    if t_m == 0:
        return 0.0, 0            # peaks at publication -> not a sleeping beauty
    c_m = C[t_m]; c_0 = C[0]
    B = 0.0
    for t in range(t_m + 1):
        den = C[t] if C[t] != 0.0 else 1.0
        B += ((c_m - c_0) / t_m * t + c_0 - C[t]) / den
    norm = np.sqrt((c_m - c_0) ** 2 + t_m * t_m)
    T = 0; dmax = -1.0
    for t in range(t_m + 1):
        d = abs((c_m - c_0) * t + (c_0 - C[t]) * t_m) / norm
        if d > dmax:
            dmax = d; T = t
    return B, T

@njit(parallel=True)
def sb_from_age_csr(ptr, ages, cnts, SBB, SBT, NC):
    """For each work w, its citation ages are ages[ptr[w]:ptr[w+1]] with counts cnts[...]."""
    n = len(ptr) - 1
    for w in prange(n):
        a0 = ptr[w]; a1 = ptr[w + 1]
        if a1 == a0:
            continue
        mx = -1; tot = 0
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                tot += cnts[j]
                if ag > mx:
                    mx = ag
        if mx < 0:
            continue
        C = np.zeros(mx + 1, np.float64)
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                C[ag] += cnts[j]
        B, T = _sb_one(C)
        SBB[w] = B; SBT[w] = T; NC[w] = tot

## 1. Age histogram per cited application -> kernel

In [ ]:
%%time
con = ps.connect()
a = con.execute(f"""SELECT cited_id, age, count(*) AS n FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE}
                    GROUP BY 1, 2 ORDER BY 1, 2""").fetchnumpy()
con.close()
cited = a['cited_id'].astype(np.int64); ages = a['age'].astype(np.int32); cnts = a['n'].astype(np.int64); del a
uni, first = np.unique(cited, return_index=True)
n = len(uni)
ptr = np.zeros(n + 1, np.int64); ptr[1:-1] = first[1:]; ptr[-1] = len(cited)
SBB = np.full(n, np.nan, np.float64); SBT = np.full(n, -1, np.int32); NC = np.zeros(n, np.int64)
sb_from_age_csr(ptr, ages, cnts, SBB, SBT, NC)
print(f'{n:,} cited applications, {int(cnts.sum()):,} citation rows -> SB computed')

## 2. Save

In [ ]:
mask = NC > 0
out = pd.DataFrame({'appln_id': uni[mask], 'SB_B': SBB[mask].astype(np.float32), 'SB_T': SBT[mask], 'n_cite': NC[mask]})
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows)')
print('SB_B summary (n_cite >= 50):'); print(out[out.n_cite >= 50]['SB_B'].describe().round(3).to_string())
display(out[out.n_cite >= 100].nlargest(10, 'SB_B'))